# 05c — Validação final do módulo de risco

Este notebook encerra a etapa experimental de validação das regras de risco.

Foram realizadas duas etapas anteriores:

- `05_validacao_risco.ipynb`: primeira seleção de 50 cenários;
- `05b_complemento_validacao_risco.ipynb`: busca de candidatos substitutos utilizando ground truth + IoU.

Na primeira seleção:

```text
50 cenários iniciais
- 16 excluídos
= 34 cenários válidos
```

Depois foram selecionados:

```text
16 novos cenários válidos
```

Portanto:

```text
34 + 16 = 50 cenários finais
```

## Objetivo

Comparar definitivamente:

```text
Regra V1
classe + posição horizontal + tamanho

versus

Regra V2
classe + posição horizontal + tamanho + posição vertical
```

utilizando o mesmo gabarito humano.

Ao final serão calculados:

- Accuracy;
- Precision macro;
- Recall macro;
- F1 macro;
- F1 weighted;
- métricas por nível de risco;
- matrizes de confusão;
- desempenho por classe de objeto;
- análise das discordâncias V1 × V2.

> Após esta etapa, os parâmetros das regras não devem mais ser alterados com base nesses 50 cenários.

## 1. Importações e localização da raiz do projeto

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def localizar_raiz_projeto() -> Path:
    atual = Path.cwd().resolve()

    for candidata in [atual, *atual.parents]:
        if (
            (candidata / "requirements.txt").exists()
            and (candidata / "src").exists()
        ):
            return candidata

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )


RAIZ_PROJETO = localizar_raiz_projeto()

if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

print("Raiz do projeto:", RAIZ_PROJETO)

## 2. Importar métricas

In [ ]:
try:
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        precision_recall_fscore_support,
    )
except ImportError:
    import subprocess

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "scikit-learn"]
    )

    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        precision_recall_fscore_support,
    )

print("Scikit-learn importado com sucesso.")

## 3. Caminhos

In [ ]:
PASTA_TABELAS = (
    RAIZ_PROJETO
    / "reports"
    / "tables"
)

PASTA_FIGURAS = (
    RAIZ_PROJETO
    / "reports"
    / "figures"
    / "validacao_risco_final"
)

CAMINHO_GABARITO_ORIGINAL = (
    PASTA_TABELAS
    / "validacao_manual_risco.csv"
)

CAMINHO_COMPLEMENTARES = (
    PASTA_TABELAS
    / "candidatos_complementares_selecionados_16.csv"
)

CAMINHO_FINAL_50 = (
    PASTA_TABELAS
    / "validacao_risco_final_50_cenarios.csv"
)

CAMINHO_METRICAS_GERAIS = (
    PASTA_TABELAS
    / "metricas_validacao_risco_final.csv"
)

CAMINHO_METRICAS_CLASSE = (
    PASTA_TABELAS
    / "metricas_validacao_risco_final_por_classe.csv"
)

CAMINHO_DISCORDANCIAS = (
    PASTA_TABELAS
    / "discordancias_validacao_risco_final.csv"
)

PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)

for caminho in [
    CAMINHO_GABARITO_ORIGINAL,
    CAMINHO_COMPLEMENTARES,
]:
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo obrigatório não encontrado:\n{caminho}"
        )

print("Gabarito original:")
print(CAMINHO_GABARITO_ORIGINAL)
print()
print("16 complementares:")
print(CAMINHO_COMPLEMENTARES)

# 4. Regras V1 e V2 congeladas

Os parâmetros abaixo são os mesmos utilizados nas etapas anteriores.

## V1

```text
peso da classe
+ posição horizontal
+ tamanho relativo
```

## V2

```text
V1
+ ajuste pela posição vertical
```

Não altere estes valores nesta etapa.

In [ ]:
PESO_OBJETO = {
    "bed": 3,
    "sofa": 3,
    "chair": 3,
    "table": 3,
    "wardrobe": 3,
    "potted plant": 2,
}

PESO_POSICAO = {
    "esquerda": 1,
    "centro": 3,
    "direita": 1,
}

PESO_TAMANHO = {
    "pequeno": 1,
    "medio": 2,
    "grande": 3,
}

AJUSTE_VERTICAL = {
    "superior": -2,
    "intermediaria": -1,
    "inferior": 0,
}


def nivel_por_pontuacao(pontuacao: int) -> str:
    if pontuacao <= 4:
        return "baixo"

    if pontuacao <= 6:
        return "medio"

    return "alto"


def classificar_risco_v1(
    classe: str,
    posicao: str,
    tamanho: str,
):
    pontuacao = (
        PESO_OBJETO[classe]
        + PESO_POSICAO[posicao]
        + PESO_TAMANHO[tamanho]
    )

    return pontuacao, nivel_por_pontuacao(pontuacao)


def classificar_risco_v2(
    classe: str,
    posicao: str,
    tamanho: str,
    posicao_vertical: str,
):
    pontuacao_v1, _ = classificar_risco_v1(
        classe=classe,
        posicao=posicao,
        tamanho=tamanho,
    )

    ajuste_vertical = AJUSTE_VERTICAL[
        posicao_vertical
    ]

    pontuacao = max(
        1,
        pontuacao_v1 + ajuste_vertical,
    )

    return (
        pontuacao,
        nivel_por_pontuacao(pontuacao),
        ajuste_vertical,
    )

# 5. Ler os 34 cenários válidos da primeira etapa

O arquivo original contém 50 cenários.

Os registros com:

```text
risco_esperado = excluir
```

serão removidos.

In [ ]:
df_original = pd.read_csv(
    CAMINHO_GABARITO_ORIGINAL,
    encoding="utf-8-sig",
)

df_original["risco_esperado"] = (
    df_original["risco_esperado"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print("Cenários originais:", len(df_original))

df_original_validos = df_original[
    df_original["risco_esperado"].isin(
        ["baixo", "medio", "alto"]
    )
].copy()

print(
    "Cenários válidos da primeira etapa:",
    len(df_original_validos),
)

print(
    "Cenários removidos:",
    len(df_original) - len(df_original_validos),
)

if len(df_original_validos) != 34:
    print()
    print(
        "ATENÇÃO: eram esperados 34 cenários válidos, "
        f"mas foram encontrados {len(df_original_validos)}."
    )

## 6. Ler os 16 cenários complementares

In [ ]:
df_complementares = pd.read_csv(
    CAMINHO_COMPLEMENTARES,
    encoding="utf-8-sig",
)

df_complementares["risco_esperado"] = (
    df_complementares["risco_esperado"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

if "usar_no_gabarito" in df_complementares.columns:
    df_complementares["usar_no_gabarito"] = (
        df_complementares["usar_no_gabarito"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df_complementares = df_complementares[
        df_complementares["usar_no_gabarito"] == "sim"
    ].copy()

print(
    "Cenários complementares selecionados:",
    len(df_complementares),
)

if len(df_complementares) != 16:
    raise RuntimeError(
        "Era esperado um arquivo com exatamente "
        f"16 cenários complementares, mas foram encontrados "
        f"{len(df_complementares)}."
    )

valores_invalidos = sorted(
    set(df_complementares["risco_esperado"])
    - {"baixo", "medio", "alto"}
)

if valores_invalidos:
    raise ValueError(
        "Existem riscos esperados inválidos nos complementares: "
        f"{valores_invalidos}"
    )

# 7. Padronizar os dois conjuntos

O primeiro conjunto usa `cenario_id`.

O complementar usa `candidato_id`.

Vamos criar um identificador final único para os 50 cenários.

In [ ]:
COLUNAS_BASE = [
    "arquivo",
    "classe",
    "confianca",
    "posicao",
    "tamanho",
    "ocupacao_imagem",
    "posicao_vertical",
    "proporcao_vertical",
    "risco_esperado",
    "observacao",
]


def garantir_colunas(df, colunas):
    df = df.copy()

    for coluna in colunas:
        if coluna not in df.columns:
            df[coluna] = pd.NA

    return df


df_original_pad = garantir_colunas(
    df_original_validos,
    COLUNAS_BASE,
)

df_complementar_pad = garantir_colunas(
    df_complementares,
    COLUNAS_BASE,
)

df_original_pad["origem_validacao"] = (
    "primeira_validacao"
)

df_complementar_pad["origem_validacao"] = (
    "complemento_ground_truth_iou"
)

df_original_pad["id_origem"] = (
    df_original_pad["cenario_id"].astype(str)
)

df_complementar_pad["id_origem"] = (
    df_complementar_pad["candidato_id"].astype(str)
)

colunas_uniao = (
    [
        "id_origem",
        "origem_validacao",
    ]
    + COLUNAS_BASE
)

df_final = pd.concat(
    [
        df_original_pad[colunas_uniao],
        df_complementar_pad[colunas_uniao],
    ],
    ignore_index=True,
)

df_final.insert(
    0,
    "cenario_final_id",
    [
        f"F{i:03d}"
        for i in range(1, len(df_final) + 1)
    ],
)

print("Total final:", len(df_final))

# 8. Validações de integridade

Antes de calcular qualquer métrica, verificamos:

- exatamente 50 cenários;
- somente as seis classes físicas;
- nenhum risco esperado vazio;
- nenhuma categoria inválida;
- distribuição de classes esperada.

In [ ]:
CLASSES_ESPERADAS = {
    "bed",
    "sofa",
    "chair",
    "table",
    "wardrobe",
    "potted plant",
}

RISCOS_VALIDOS = {
    "baixo",
    "medio",
    "alto",
}

if len(df_final) != 50:
    raise RuntimeError(
        f"Esperávamos 50 cenários finais, "
        f"mas foram encontrados {len(df_final)}."
    )

classes_encontradas = set(
    df_final["classe"].dropna().astype(str)
)

classes_invalidas = (
    classes_encontradas - CLASSES_ESPERADAS
)

if classes_invalidas:
    raise ValueError(
        f"Classes inesperadas: {classes_invalidas}"
    )

riscos_encontrados = set(
    df_final["risco_esperado"]
    .dropna()
    .astype(str)
)

riscos_invalidos = (
    riscos_encontrados - RISCOS_VALIDOS
)

if riscos_invalidos:
    raise ValueError(
        f"Riscos esperados inválidos: {riscos_invalidos}"
    )

if df_final["risco_esperado"].isna().any():
    raise ValueError(
        "Existem valores vazios em risco_esperado."
    )

print("Validações básicas: OK")

## 9. Distribuição final por classe

In [ ]:
distribuicao_classe = (
    df_final["classe"]
    .value_counts()
    .rename_axis("classe")
    .reset_index(name="quantidade")
)

display(distribuicao_classe)

distribuicao_esperada = {
    "bed": 9,
    "sofa": 9,
    "chair": 8,
    "table": 8,
    "wardrobe": 8,
    "potted plant": 8,
}

for classe, quantidade in distribuicao_esperada.items():
    quantidade_real = int(
        (df_final["classe"] == classe).sum()
    )

    if quantidade_real != quantidade:
        print(
            f"ATENÇÃO: {classe} possui "
            f"{quantidade_real} cenários; "
            f"eram esperados {quantidade}."
        )

## 10. Distribuição final do gabarito de risco

In [ ]:
distribuicao_risco = (
    df_final["risco_esperado"]
    .value_counts()
    .reindex(
        ["baixo", "medio", "alto"],
        fill_value=0,
    )
    .rename_axis("risco_esperado")
    .reset_index(name="quantidade")
)

display(distribuicao_risco)

# 11. Calcular previsões V1 e V2

As duas regras são aplicadas agora aos mesmos 50 cenários finais.

In [ ]:
resultados_v1 = []
resultados_v2 = []

for _, linha in df_final.iterrows():
    pontuacao_v1, risco_v1 = (
        classificar_risco_v1(
            classe=linha["classe"],
            posicao=linha["posicao"],
            tamanho=linha["tamanho"],
        )
    )

    (
        pontuacao_v2,
        risco_v2,
        ajuste_vertical,
    ) = classificar_risco_v2(
        classe=linha["classe"],
        posicao=linha["posicao"],
        tamanho=linha["tamanho"],
        posicao_vertical=linha["posicao_vertical"],
    )

    resultados_v1.append(
        (pontuacao_v1, risco_v1)
    )

    resultados_v2.append(
        (
            pontuacao_v2,
            risco_v2,
            ajuste_vertical,
        )
    )

df_final["pontuacao_v1"] = [
    item[0]
    for item in resultados_v1
]

df_final["risco_v1"] = [
    item[1]
    for item in resultados_v1
]

df_final["pontuacao_v2"] = [
    item[0]
    for item in resultados_v2
]

df_final["risco_v2"] = [
    item[1]
    for item in resultados_v2
]

df_final["ajuste_vertical"] = [
    item[2]
    for item in resultados_v2
]

df_final["acerto_v1"] = (
    df_final["risco_v1"]
    == df_final["risco_esperado"]
)

df_final["acerto_v2"] = (
    df_final["risco_v2"]
    == df_final["risco_esperado"]
)

display(
    df_final[
        [
            "cenario_final_id",
            "classe",
            "risco_esperado",
            "risco_v1",
            "risco_v2",
            "acerto_v1",
            "acerto_v2",
        ]
    ].head(10)
)

# 12. Métricas finais gerais

In [ ]:
y_real = df_final["risco_esperado"]
y_v1 = df_final["risco_v1"]
y_v2 = df_final["risco_v2"]

ROTULOS = [
    "baixo",
    "medio",
    "alto",
]

metricas = []

for versao, previsao in [
    ("V1", y_v1),
    ("V2", y_v2),
]:
    accuracy = accuracy_score(
        y_real,
        previsao,
    )

    (
        precision_macro,
        recall_macro,
        f1_macro,
        _,
    ) = precision_recall_fscore_support(
        y_real,
        previsao,
        labels=ROTULOS,
        average="macro",
        zero_division=0,
    )

    f1_weighted = f1_score(
        y_real,
        previsao,
        labels=ROTULOS,
        average="weighted",
        zero_division=0,
    )

    metricas.append(
        {
            "versao": versao,
            "cenarios": len(df_final),
            "accuracy": accuracy,
            "precision_macro": precision_macro,
            "recall_macro": recall_macro,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
        }
    )

df_metricas = pd.DataFrame(metricas)

display(
    df_metricas.style.format(
        {
            "accuracy": "{:.2%}",
            "precision_macro": "{:.2%}",
            "recall_macro": "{:.2%}",
            "f1_macro": "{:.2%}",
            "f1_weighted": "{:.2%}",
        }
    )
)

## 13. Relatório final por nível de risco — V1

In [ ]:
relatorio_v1 = classification_report(
    y_real,
    y_v1,
    labels=ROTULOS,
    output_dict=True,
    zero_division=0,
)

df_relatorio_v1 = (
    pd.DataFrame(relatorio_v1)
    .T
    .reset_index()
    .rename(columns={"index": "nivel"})
)

display(df_relatorio_v1)

## 14. Relatório final por nível de risco — V2

In [ ]:
relatorio_v2 = classification_report(
    y_real,
    y_v2,
    labels=ROTULOS,
    output_dict=True,
    zero_division=0,
)

df_relatorio_v2 = (
    pd.DataFrame(relatorio_v2)
    .T
    .reset_index()
    .rename(columns={"index": "nivel"})
)

display(df_relatorio_v2)

# 15. Matrizes de confusão finais

In [ ]:
matriz_v1 = confusion_matrix(
    y_real,
    y_v1,
    labels=ROTULOS,
)

matriz_v2 = confusion_matrix(
    y_real,
    y_v2,
    labels=ROTULOS,
)


def plotar_matriz_confusao(
    matriz,
    titulo,
    caminho_saida,
):
    fig, ax = plt.subplots(figsize=(7, 6))

    imagem = ax.imshow(matriz)

    ax.set_xticks(
        np.arange(len(ROTULOS)),
        labels=ROTULOS,
    )

    ax.set_yticks(
        np.arange(len(ROTULOS)),
        labels=ROTULOS,
    )

    ax.set_xlabel("Previsto")
    ax.set_ylabel("Esperado")
    ax.set_title(titulo)

    for i in range(matriz.shape[0]):
        for j in range(matriz.shape[1]):
            ax.text(
                j,
                i,
                str(matriz[i, j]),
                ha="center",
                va="center",
            )

    fig.colorbar(imagem, ax=ax)
    plt.tight_layout()

    fig.savefig(
        caminho_saida,
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()


plotar_matriz_confusao(
    matriz_v1,
    "Matriz de confusão final — Regra V1",
    PASTA_FIGURAS / "matriz_confusao_risco_v1.png",
)

In [ ]:
plotar_matriz_confusao(
    matriz_v2,
    "Matriz de confusão final — Regra V2",
    PASTA_FIGURAS / "matriz_confusao_risco_v2.png",
)

# 16. Desempenho final por classe de objeto

In [ ]:
desempenho_classe = (
    df_final
    .groupby("classe")
    .agg(
        cenarios=("cenario_final_id", "count"),
        accuracy_v1=("acerto_v1", "mean"),
        accuracy_v2=("acerto_v2", "mean"),
    )
    .reset_index()
)

desempenho_classe["diferenca_v2_v1"] = (
    desempenho_classe["accuracy_v2"]
    - desempenho_classe["accuracy_v1"]
)

display(
    desempenho_classe.style.format(
        {
            "accuracy_v1": "{:.2%}",
            "accuracy_v2": "{:.2%}",
            "diferenca_v2_v1": "{:+.2%}",
        }
    )
)

# 17. Casos em que V1 e V2 discordam

Esta análise mostra exatamente onde a inclusão da posição vertical altera a decisão.

In [ ]:
df_discordancias = df_final[
    df_final["risco_v1"]
    != df_final["risco_v2"]
].copy()

if df_discordancias.empty:
    print(
        "V1 e V2 produziram a mesma classificação "
        "em todos os 50 cenários."
    )
else:
    df_discordancias["somente_v1_acertou"] = (
        df_discordancias["acerto_v1"]
        & ~df_discordancias["acerto_v2"]
    )

    df_discordancias["somente_v2_acertou"] = (
        ~df_discordancias["acerto_v1"]
        & df_discordancias["acerto_v2"]
    )

    df_discordancias["ambas_erraram"] = (
        ~df_discordancias["acerto_v1"]
        & ~df_discordancias["acerto_v2"]
    )

    display(
        df_discordancias[
            [
                "cenario_final_id",
                "id_origem",
                "classe",
                "posicao",
                "tamanho",
                "posicao_vertical",
                "risco_esperado",
                "risco_v1",
                "risco_v2",
                "acerto_v1",
                "acerto_v2",
            ]
        ]
    )

    print(
        "Total de discordâncias:",
        len(df_discordancias),
    )

    print(
        "Somente V1 acertou:",
        int(
            df_discordancias[
                "somente_v1_acertou"
            ].sum()
        ),
    )

    print(
        "Somente V2 acertou:",
        int(
            df_discordancias[
                "somente_v2_acertou"
            ].sum()
        ),
    )

    print(
        "Ambas erraram:",
        int(
            df_discordancias[
                "ambas_erraram"
            ].sum()
        ),
    )

# 18. Análise específica dos casos de risco ALTO

Para a proposta assistiva, é importante observar se a regra consegue reconhecer situações de maior risco.

Nesta seção calculamos:

- quantos casos `alto` existem no gabarito;
- quantos foram corretamente identificados por V1;
- quantos foram corretamente identificados por V2;
- falsos alertas de nível `alto`.

In [ ]:
total_alto_real = (
    df_final["risco_esperado"] == "alto"
).sum()

verdadeiros_altos_v1 = (
    (df_final["risco_esperado"] == "alto")
    & (df_final["risco_v1"] == "alto")
).sum()

verdadeiros_altos_v2 = (
    (df_final["risco_esperado"] == "alto")
    & (df_final["risco_v2"] == "alto")
).sum()

falsos_altos_v1 = (
    (df_final["risco_esperado"] != "alto")
    & (df_final["risco_v1"] == "alto")
).sum()

falsos_altos_v2 = (
    (df_final["risco_esperado"] != "alto")
    & (df_final["risco_v2"] == "alto")
).sum()

resumo_alto = pd.DataFrame(
    [
        {
            "versao": "V1",
            "casos_altos_reais": total_alto_real,
            "altos_corretamente_identificados": verdadeiros_altos_v1,
            "falsos_alertas_alto": falsos_altos_v1,
        },
        {
            "versao": "V2",
            "casos_altos_reais": total_alto_real,
            "altos_corretamente_identificados": verdadeiros_altos_v2,
            "falsos_alertas_alto": falsos_altos_v2,
        },
    ]
)

display(resumo_alto)

# 19. Comparação final V1 × V2

In [ ]:
linha_v1 = df_metricas[
    df_metricas["versao"] == "V1"
].iloc[0]

linha_v2 = df_metricas[
    df_metricas["versao"] == "V2"
].iloc[0]

print("RESULTADO FINAL")
print("=" * 50)
print()

print(
    f'Accuracy V1: {linha_v1["accuracy"]:.2%}'
)

print(
    f'Accuracy V2: {linha_v2["accuracy"]:.2%}'
)

print(
    f'Diferença: '
    f'{linha_v2["accuracy"] - linha_v1["accuracy"]:+.2%}'
)

print()

print(
    f'F1 macro V1: {linha_v1["f1_macro"]:.2%}'
)

print(
    f'F1 macro V2: {linha_v2["f1_macro"]:.2%}'
)

print(
    f'Diferença: '
    f'{linha_v2["f1_macro"] - linha_v1["f1_macro"]:+.2%}'
)

print()
print("=" * 50)

if (
    linha_v2["accuracy"] > linha_v1["accuracy"]
    and linha_v2["f1_macro"] >= linha_v1["f1_macro"]
):
    print(
        "CONCLUSÃO PRELIMINAR: "
        "a Regra V2 apresentou desempenho superior "
        "nos principais indicadores."
    )

elif (
    linha_v1["accuracy"] > linha_v2["accuracy"]
    and linha_v1["f1_macro"] >= linha_v2["f1_macro"]
):
    print(
        "CONCLUSÃO PRELIMINAR: "
        "a Regra V1 apresentou desempenho superior "
        "nos principais indicadores."
    )

else:
    print(
        "CONCLUSÃO PRELIMINAR: "
        "os resultados são mistos. "
        "Consulte as matrizes de confusão e "
        "a análise dos casos de maior risco."
    )

# 20. Salvar conjunto final e métricas

Estes arquivos serão os resultados definitivos da validação do módulo de risco.

In [ ]:
df_final.to_csv(
    CAMINHO_FINAL_50,
    index=False,
    encoding="utf-8-sig",
)

df_metricas.to_csv(
    CAMINHO_METRICAS_GERAIS,
    index=False,
    encoding="utf-8-sig",
)

desempenho_classe.to_csv(
    CAMINHO_METRICAS_CLASSE,
    index=False,
    encoding="utf-8-sig",
)

df_discordancias.to_csv(
    CAMINHO_DISCORDANCIAS,
    index=False,
    encoding="utf-8-sig",
)

df_relatorio_v1.to_csv(
    PASTA_TABELAS
    / "relatorio_classificacao_risco_v1_final.csv",
    index=False,
    encoding="utf-8-sig",
)

df_relatorio_v2.to_csv(
    PASTA_TABELAS
    / "relatorio_classificacao_risco_v2_final.csv",
    index=False,
    encoding="utf-8-sig",
)

resumo_alto.to_csv(
    PASTA_TABELAS
    / "resumo_risco_alto_v1_v2_final.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Arquivos finais salvos.")
print()
print("Conjunto final de 50 cenários:")
print(CAMINHO_FINAL_50)
print()
print("Métricas:")
print(CAMINHO_METRICAS_GERAIS)
print()
print("Figuras:")
print(PASTA_FIGURAS)

# 21. Resumo metodológico para o TCC

A avaliação final separa duas tarefas distintas.

## Detecção de objetos

O YOLOv8n é avaliado por métricas próprias de detecção:

```text
Precision
Recall
mAP@50
mAP@50-95
```

## Classificação do potencial de impacto na locomoção

O módulo baseado em regras é avaliado contra o gabarito humano:

```text
Accuracy
Precision
Recall
F1-score
Matriz de confusão
```

O conjunto final contém 50 cenários válidos distribuídos entre:

```text
bed
sofa
chair
table
wardrobe
potted plant
```

Os 16 casos substitutos foram selecionados somente após confirmação da classe pelo ground truth e correspondência espacial entre anotação e previsão com:

```text
IoU >= 0,50
```

Isso evita contabilizar erros de detecção como erros do módulo de risco.

## Próxima etapa

Após analisar os resultados deste notebook e selecionar a regra final:

```text
YOLO
    ↓
regra de risco final
    ↓
baixo / médio / alto
    ↓
geração de alerta
    ↓
saída em áudio
```

# 22. Importante

Depois de executar este notebook:

**não ajuste mais os parâmetros V1/V2 utilizando estes 50 cenários.**

Caso a V2 seja escolhida, seus parâmetros devem ser tratados como congelados para a versão final do protótipo.

Isso evita adaptar a regra repetidamente ao próprio conjunto usado para medir seu desempenho.